In [0]:
from pyspark.sql import *

In [0]:
df = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
            .load("/Volumes/pyspark/source/ff_source/BigMart Sales.csv")
df.display()

###WINDOWS_FUNCTIONS


####ROW_NUMBER
It will used to create a new column with sequence of number (similar to SQL ROW_NUMBER function)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *


In [0]:

df.withColumn('rowcol', row_number().over(Window.orderBy(col('Item_Identifier')))).display()

####RANK AND DENSE_RANK

It is used to provide a rank to any column in df

RANK: It will skip the rank, if there is any match in column values

DENSE_Rank: It wont skip the rank

In [0]:
df.withColumn('rank',rank().over(Window.orderBy(col('Item_Identifier')))).display()

In [0]:
df.withColumn('dense_rank',dense_rank().over(Window.orderBy(col('Item_Identifier')))).display()

In [0]:
df.withColumn('row_number',row_number().over(Window.orderBy(col('Item_Identifier'))))\
    .withColumn('rank',rank().over(Window.orderBy(col('Item_Identifier'))))\
    .withColumn('dense_rank',dense_rank().over(Window.orderBy(col('Item_Identifier')))).display()

In [0]:
df.withColumn('rank', rank().over(Window().orderBy(col('Item_Identifier').desc()))).select('Item_Identifier','rank').display()

####CUMULATIVE_SUM

The cumulative sum calculates the total of a column's values up to the current row, often ordered by a specific column (like date).

In [0]:
#By default without frame clause in over() it will add all the values in column
df.withColumn('cumsum', sum('Item_MRP').over(Window.partitionBy(col('Item_Type')))).display()

In [0]:
#with frame clause (rowbetween with Window.unboundedPreceding and Window.currentRow)
df.withColumn('cumsum',sum('Item_MRP').over(Window.partitionBy(col('Item_Type')).rowsBetween(Window.unboundedPreceding,Window.currentRow))).select('Item_Type','Item_MRP','cumsum').filter((col('Item_Type') == 'Dairy') & (col('Item_MRP') > 100)).display()

In [0]:
#with frame clause (rowbetween with Window.unboundedPreceding and Window.unboundedFollowing) it will give total sum not cumsum
df.withColumn('totalsum',sum('Item_MRP').over(Window.rowsBetween(Window.unboundedPreceding,Window.unboundedFollowing)))\
    .select('Item_Type','Item_MRP','totalsum').display()
